In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

processed_data_path = "../data/processed/"

processed_data = pd.read_csv(os.path.join(processed_data_path, "processed_data.csv"))


In [2]:
X = processed_data.drop(columns=["Churn"])
y = processed_data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Import Baseline Model

In [3]:
baseline_metrics = json.load(open("../models/baseline/baseline_random_forest_metrics.json"))
baseline_model = joblib.load("../models/baseline/baseline_random_forest.joblib")
baseline_metrics

{'Model': 'RandomForestClassifier(default)',
 'Accuracy': 0.7984386089425124,
 'Precision': 0.663003663003663}

## Tuning Baseline Model 

### - Using optuna

In [4]:
import optuna
from sklearn.metrics import accuracy_score, precision_score


def objective(trial):
    params = {
        # 1. จำนวนต้นไม้ในป่า
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        # 2. เกณฑ์วัดความบริสุทธิ์
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        # 3. ความลึกสูงสุดของแต่ละต้น
        'max_depth': trial.suggest_int('max_depth', 3, 32),
        # 4. จำนวนข้อมูลขั้นต่ำในการแตกกิ่ง
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 40),
        # 5. จำนวนข้อมูลขั้นต่ำที่ใบ
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 32),
        # 6. จำนวน Feature ที่สุ่มมาใช้ในแต่ละต้น
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        # 7. สัดส่วนข้อมูล bootstrap ต่อต้น
        'max_samples': trial.suggest_float('max_samples', 0.5, 1.0),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
    }

    model = RandomForestClassifier(**params)

    # --- 5-Fold Cross-Validation (train set เท่านั้น) ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    return scores.mean()

In [5]:
# สั่งให้ Optuna รันการค้นหา
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-09-07 18:06:27,516] A new study created in memory with name: no-name-e57bdc5b-9b09-4047-9211-1af2903849b8
[I 2026-09-07 18:06:29,533] Trial 0 finished with value: 0.8403979252449 and parameters: {'n_estimators': 67, 'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 13, 'max_features': 'sqrt', 'max_samples': 0.6900232357964644}. Best is trial 0 with value: 0.8403979252449.
[I 2026-09-07 18:06:31,909] Trial 1 finished with value: 0.8410101109276804 and parameters: {'n_estimators': 224, 'criterion': 'entropy', 'max_depth': 31, 'min_samples_split': 39, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_samples': 0.9314686945223819}. Best is trial 1 with value: 0.8410101109276804.
[I 2026-09-07 18:06:33,590] Trial 2 finished with value: 0.8411985391530037 and parameters: {'n_estimators': 208, 'criterion': 'entropy', 'max_depth': 25, 'min_samples_split': 26, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'max_samples': 0.9490811078166468}. Best is t

In [10]:
# แสดงผลลัพธ์
print("="*40)
print(f" Accuracy สูงสุดที่ Optuna หาได้: {study.best_value:.4f}")
print(" ค่า Hyperparameters ที่ดีที่สุดของ Decision Tree:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")
print("="*40)

 Accuracy สูงสุดที่ Optuna หาได้: 0.8421
 ค่า Hyperparameters ที่ดีที่สุดของ Decision Tree:
  - n_estimators: 298
  - criterion: entropy
  - max_depth: 17
  - min_samples_split: 22
  - min_samples_leaf: 18
  - max_features: sqrt
  - max_samples: 0.9343730522395282


In [11]:
# สร้างโมเดลด้วย best_params จาก study
best_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)

# เทรนกับ Train Set
best_model.fit(X_train, y_train)

# ประเมินกับ Test Set
y_pred = best_model.predict(X_test)

print(f"Baseline  Accuracy (Test): {baseline_metrics['Accuracy']:.4f}")
print(f"Tuned     Accuracy (Test): {accuracy_score(y_test, y_pred):.4f}")
print(f"Tuned     Precision (Test): {precision_score(y_test, y_pred):.4f}")

Baseline  Accuracy (Test): 0.7984
Tuned     Accuracy (Test): 0.8119
Tuned     Precision (Test): 0.7015


In [12]:
# ดูจำนวนและสัดส่วนของ Target ในข้อมูล
print("จำนวนข้อมูลแยกตาม Class:")
print(pd.Series(y_test).value_counts())

print("\nสัดส่วน % ของแต่ละ Class:")
print(pd.Series(y_test).value_counts(normalize=True) * 100)

จำนวนข้อมูลแยกตาม Class:
Churn
0    1036
1     373
Name: count, dtype: int64

สัดส่วน % ของแต่ละ Class:
Churn
0    73.527324
1    26.472676
Name: proportion, dtype: float64


In [13]:
from sklearn.metrics import classification_report

# Predict ผลลัพธ์
y_pred = best_model.predict(X_test)

# ดูรายงานสรุป
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.92      0.88      1036
           1       0.70      0.50      0.59       373

    accuracy                           0.81      1409
   macro avg       0.77      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409

